### Imports

In [1]:
library(tidyverse)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.5.1
✔ ggplot2   4.0.1     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


### Datos

In [3]:
## Microdatos de la ENUT 2024
path = "/home/luisdx/Documents/dat4ccion/enut_2024_bd_csv/tsdem.csv"
TSDEM <- read.csv(path)
path = "/home/luisdx/Documents/dat4ccion/enut_2024_bd_csv/tmodulo.csv"
TMODULO <- read.csv(path)
path = "/home/luisdx/Documents/dat4ccion/enut_2024_bd_csv/tvar_crea.csv"
TVAR_CREA <- read.csv(path)

### Procesamiento: La Doble Jornada

In [4]:
codigos_hijos <- c(3, 4)
codigos_padres <- c(1,2)

df_aux_hijxs <- TSDEM %>%
  group_by(LLAVEHOG) %>%
  mutate(
    hogar_con_hijos = any(PAREN %in% codigos_hijos),
    TIENE_HIJOS = ifelse(hogar_con_hijos & PAREN %in% codigos_padres, 1, 0)
  ) %>%
  ungroup() %>%
  select(LLAVESDE, TIENE_HIJOS)

codigo_union <- c(1, 5)
df_aux_civil <- TMODULO %>%
  mutate(
    EST_CONY = ifelse(P4_5 %in% codigo_union, 1, 0)
  ) %>%
  select(LLAVEMOD, EST_CONY)

df <- TVAR_CREA %>%
  filter(EDAD >= 18) %>%
  left_join(df_aux_hijxs, by = c("LLAVEMOD" = "LLAVESDE")) %>% 
  left_join(df_aux_civil, by = "LLAVEMOD") %>% 
  mutate(
    GENERO = case_when(
      SEXO == 2 & ((TIENE_HIJOS == 1) | (EST_CONY == 1))  ~ "Mujer con pareja y/o descendencia",
      SEXO == 2 & !((TIENE_HIJOS == 1) | (EST_CONY == 1)) ~ "Mujer sin pareja ni descendencia",
      SEXO == 1 & ((TIENE_HIJOS == 1) | (EST_CONY == 1))  ~ "Hombre Unido y/o con Hij@s y/o con Niet@s",
      SEXO == 1 & !((TIENE_HIJOS == 1) | (EST_CONY == 1)) ~ "Hombre con pareja y/o descendencia",
      TRUE ~ NA_character_
    )
  ) %>%
  filter(!is.na(GENERO) & GENERO != "No especificado")

df$GENERACION <- cut(
  df$EDAD,
  breaks = c(-Inf, 30, 45, 60, Inf),
  labels = c("Menores de 30 años", "De 30 a 45 años", "De 45 a 60 años", "Mayores de 60 años"),
  right = FALSE
)
df$TRAB_NO_REM_TOTAL <- df$TRAB_NO_REM_HOG + df$TRAB_NO_REM_CON_CP

resumen <- df %>%
  group_by(GENERO, GENERACION) %>%
  summarise(
    Trabajo_Remunerado = weighted.mean(ACTIV_MERC, FAC_PER, na.rm = TRUE)/7,
    Trabajo_No_Remunerado = weighted.mean(TRAB_NO_REM_TOTAL, FAC_PER, na.rm = TRUE)/7,
    Tiempo_de_Ocio = weighted.mean(ACTIV_CONVIV, FAC_PER, na.rm = TRUE)/7, 
    Tiempo_de_Estudio = weighted.mean(ACTIV_ESTUD, FAC_PER, na.rm = TRUE)/7,  
    Tiempo_de_Autocuidado = weighted.mean(ACTIV_CUID_PER, FAC_PER, na.rm = TRUE)/7,
    Tiempo_Personal = weighted.mean(ACTIV_CONVIV+ACTIV_ESTUD+ACTIV_CUID_PER, FAC_PER, na.rm = TRUE)/7,
  ) %>%
  pivot_longer(cols = -c(GENERO, GENERACION), names_to = "TIEMPO", values_to = "HORAS")

`summarise()` has grouped output by 'GENERO'. You can override using the
`.groups` argument.


In [5]:
resumen

GENERO,GENERACION,TIEMPO,HORAS
<chr>,<fct>,<chr>,<dbl>
Hombre Unido y/o con Hij@s y/o con Niet@s,Menores de 30 años,Trabajo_Remunerado,7.790495771
Hombre Unido y/o con Hij@s y/o con Niet@s,Menores de 30 años,Trabajo_No_Remunerado,4.115720956
Hombre Unido y/o con Hij@s y/o con Niet@s,Menores de 30 años,Tiempo_de_Ocio,3.225646801
Hombre Unido y/o con Hij@s y/o con Niet@s,Menores de 30 años,Tiempo_de_Estudio,0.119164464
Hombre Unido y/o con Hij@s y/o con Niet@s,Menores de 30 años,Tiempo_de_Autocuidado,9.621450237
Hombre Unido y/o con Hij@s y/o con Niet@s,Menores de 30 años,Tiempo_Personal,12.966261502
Hombre Unido y/o con Hij@s y/o con Niet@s,De 30 a 45 años,Trabajo_Remunerado,7.807943075
Hombre Unido y/o con Hij@s y/o con Niet@s,De 30 a 45 años,Trabajo_No_Remunerado,4.046326760
Hombre Unido y/o con Hij@s y/o con Niet@s,De 30 a 45 años,Tiempo_de_Ocio,2.995270508


### Procesamiento: Brecha salarial desagregado por estado

In [ ]:
PATH <- "C:/Users/semiramis/Documents/Me/Dat4ccion/"
source(paste0(PATH, "/syntax/00-Inicio_Proyecto.R"))

library(tidyverse)
library(survey)

muestra_2024 <- readRDS(paste0(DATADIR, "muestra_2024.rds")) # 308598
muestra_inicial_2024 <- readRDS(paste0(DATADIR, "muestra_inicial_2024.rds")) # 135675

# unimos
vars_faltantes <- setdiff(names(muestra_2024), names(muestra_inicial_2024))

muestra_inicial_2024 <- muestra_inicial_2024 %>%   # 135675, 54
  left_join(
    muestra_2024 %>%
      select(id_persona, id_hogar, all_of(vars_faltantes)),
    by = c("id_persona", "id_hogar"))

# agrupacion nivel_edu
muestra_inicial_2024 <- muestra_inicial_2024 %>%
  mutate(
    nivel_edu = case_when(
      nivelaprob %in% c(0,1) ~ "Sin escolaridad / Preescolar",
      nivelaprob == 2 ~ "Primaria",
      nivelaprob == 3 ~ "Secundaria",
      nivelaprob %in% c(4,5,6) ~ "Media superior",
      nivelaprob %in% c(7) ~ "Superior",
      nivelaprob %in% c(8,9,10) ~ "Posgrado",
      TRUE ~ NA_character_))

muestra_inicial_2024$nivel_edu <- factor(
  muestra_inicial_2024$nivel_edu,
  levels = c(
    "Sin escolaridad / Preescolar",
    "Primaria",
    "Secundaria",
    "Media superior",
    "Superior",
    "Posgrado"))


# Diseño muestral
diseno <- svydesign(ids = ~1, data = muestra_inicial_2024, weights = ~factor)

# G1 — Scorecard: ingreso promedio ponderado por sexo
g1 <- svyby(~ingreso_mensual, ~sexo, diseno, svymean, na.rm = TRUE)
brecha <- (g1$ingreso_mensual[g1$sexo==2] /
             g1$ingreso_mensual[g1$sexo==1]) * 100

readr::write_csv(g1, paste0(OUTPUTDIR,"ENIGH_Brecha_ing_sex.csv"))

# G2 — Ingreso por nivel educativo y sexo
g2 <- svyby(~ingreso_mensual, ~sexo + nivel_edu, diseno, svymean, na.rm = TRUE)

g2$sexo <- factor(
  g2$sexo,
  levels = c(1,2),
  labels = c("Hombre","Mujer"))

### Procesamiento: Razón de Trabajo No Remunerado mujeres vs hombres desagregado por Estado

In [6]:
df <- TVAR_CREA %>%
  filter(EDAD >= 18) 
df$TRAB_NO_REM_TOTAL <- df$TRAB_NO_REM_HOG + df$TRAB_NO_REM_CON_CP
resumen <- df %>%
  group_by(SEXO, CVE_ENT) %>%
  summarise(
    TOTAL_TNR = weighted.mean(TRAB_NO_REM_TOTAL, FAC_PER, na.rm = TRUE)/7,
    .groups = "drop"
  ) 
resumen <- resumen %>%
  pivot_wider(
      names_from = SEXO,
      values_from = TOTAL_TNR,
    names_prefix = "SEXO_"
  ) %>%
  mutate(
    RATIO_TNR = SEXO_2 / SEXO_1
  )
cat_entidades <- data.frame(
  CVE_ENT_INT = 1:32,
  ESTADO = c("Aguascalientes", "Baja California", "Baja California Sur", "Campeche", 
             "Coahuila de Zaragoza", "Colima", "Chiapas", "Chihuahua", "Ciudad de México", 
             "Durango", "Guanajuato", "Guerrero", "Hidalgo", "Jalisco", "Estado de México", 
             "Michoacán de Ocampo", "Morelos", "Nayarit", "Nuevo León", "Oaxaca", 
             "Puebla", "Querétaro", "Quintana Roo", "San Luis Potosí", "Sinaloa", 
             "Sonora", "Tabasco", "Tamaulipas", "Tlaxcala", "Veracruz de Ignacio de la Llave", 
             "Yucatán", "Zacatecas"),
  ISO_CODE = c("MX-AGU", "MX-BCN", "MX-BCS", "MX-CAM", "MX-COA", "MX-COL", "MX-CHP", 
               "MX-CHH", "MX-CMX", "MX-DUR", "MX-GUA", "MX-GRO", "MX-HID", "MX-JAL", 
               "MX-MEX", "MX-MIC", "MX-MOR", "MX-NAY", "MX-NLE", "MX-OAX", "MX-PUE", 
               "MX-QUE", "MX-ROO", "MX-SLP", "MX-SIN", "MX-SON", "MX-TAB", "MX-TAM", 
               "MX-TLA", "MX-VER", "MX-YUC", "MX-ZAC")
)
resumen_final <- resumen %>%
  rename(CVE_ENT_INT = CVE_ENT) %>% 
  left_join(cat_entidades, by = "CVE_ENT_INT") %>%
  mutate(CVE_ENT_STR = str_pad(CVE_ENT_INT, width = 2, pad = "0"))
df_looker <- resumen_final %>%
  pivot_longer(
    cols = RATIO_TNR, 
    names_to = "KPI", 
    values_to = "VALOR_KPI"
  ) %>%
  select(CVE_ENT_INT, ESTADO, ISO_CODE, KPI, VALOR_KPI)

In [7]:
df_looker

CVE_ENT_INT,ESTADO,ISO_CODE,KPI,VALOR_KPI
<int>,<chr>,<chr>,<chr>,<dbl>
1,Aguascalientes,MX-AGU,RATIO_TNR,2.251906
2,Baja California,MX-BCN,RATIO_TNR,1.799851
3,Baja California Sur,MX-BCS,RATIO_TNR,1.999925
4,Campeche,MX-CAM,RATIO_TNR,2.282772
5,Coahuila de Zaragoza,MX-COA,RATIO_TNR,2.265615
6,Colima,MX-COL,RATIO_TNR,2.175710
7,Chiapas,MX-CHP,RATIO_TNR,2.706640
8,Chihuahua,MX-CHH,RATIO_TNR,2.157134
9,Ciudad de México,MX-CMX,RATIO_TNR,1.903961


### Procesamiento: Brecha Salarial Desagregado por Estado

In [2]:
path = "/home/luisdx/Documents/dat4ccion/enigh_2024_bv_csv/muestra_inicial_2024.csv"
MUESTRA <- read.csv(path)
MUESTRA$SEXO <- MUESTRA$sexo
MUESTRA$ENTIDAD <- MUESTRA$entidad
resumen <- MUESTRA %>%
  group_by(SEXO, ENTIDAD) %>%
  summarise(
    INGRESO_MENSUAL_ENTIDAD = weighted.mean(ingreso_mensual, factor, na.rm = TRUE),
    .groups = "drop"
  ) 
resumen <- resumen %>%
  pivot_wider(
      names_from = SEXO,
      values_from = INGRESO_MENSUAL_ENTIDAD,
    names_prefix = "SEXO_"
  ) %>%
  mutate(
    BRECHA_GEN = (SEXO_1 - SEXO_2) / SEXO_1
)
cat_entidades <- data.frame(
  CVE_ENT_INT = 1:32,
  ESTADO = c("Aguascalientes", "Baja California", "Baja California Sur", "Campeche", 
             "Coahuila de Zaragoza", "Colima", "Chiapas", "Chihuahua", "Ciudad de México", 
             "Durango", "Guanajuato", "Guerrero", "Hidalgo", "Jalisco", "Estado de México", 
             "Michoacán de Ocampo", "Morelos", "Nayarit", "Nuevo León", "Oaxaca", 
             "Puebla", "Querétaro", "Quintana Roo", "San Luis Potosí", "Sinaloa", 
             "Sonora", "Tabasco", "Tamaulipas", "Tlaxcala", "Veracruz de Ignacio de la Llave", 
             "Yucatán", "Zacatecas"),
  ISO_CODE = c("MX-AGU", "MX-BCN", "MX-BCS", "MX-CAM", "MX-COA", "MX-COL", "MX-CHP", 
               "MX-CHH", "MX-CMX", "MX-DUR", "MX-GUA", "MX-GRO", "MX-HID", "MX-JAL", 
               "MX-MEX", "MX-MIC", "MX-MOR", "MX-NAY", "MX-NLE", "MX-OAX", "MX-PUE", 
               "MX-QUE", "MX-ROO", "MX-SLP", "MX-SIN", "MX-SON", "MX-TAB", "MX-TAM", 
               "MX-TLA", "MX-VER", "MX-YUC", "MX-ZAC")
)
resumen_final <- resumen %>%
  rename(CVE_ENT_INT = ENTIDAD) %>% 
  left_join(cat_entidades, by = "CVE_ENT_INT") %>%
  mutate(CVE_ENT_STR = str_pad(CVE_ENT_INT, width = 2, pad = "0"))
df_looker <- resumen_final %>%
  pivot_longer(
    cols = BRECHA_GEN, 
    names_to = "KPI", 
    values_to = "VALOR_KPI"
  ) %>%
  select(CVE_ENT_INT, ESTADO, ISO_CODE, KPI, VALOR_KPI)

In [6]:
df_looker

CVE_ENT_INT,ESTADO,ISO_CODE,KPI,VALOR_KPI
<int>,<chr>,<chr>,<chr>,<dbl>
1,Aguascalientes,MX-AGU,BRECHA_GEN,0.2732940
2,Baja California,MX-BCN,BRECHA_GEN,0.2615228
3,Baja California Sur,MX-BCS,BRECHA_GEN,0.2360659
4,Campeche,MX-CAM,BRECHA_GEN,0.3452751
5,Coahuila de Zaragoza,MX-COA,BRECHA_GEN,0.3086182
6,Colima,MX-COL,BRECHA_GEN,0.2752620
7,Chiapas,MX-CHP,BRECHA_GEN,0.3393496
8,Chihuahua,MX-CHH,BRECHA_GEN,0.2805027
9,Ciudad de México,MX-CMX,BRECHA_GEN,0.2384701
